## 2. Khoảng cách và láng giềng (e1)

In [5]:
"""Bước 1: tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f"  d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"  d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hoá bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
df = pd.read_csv("sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)
print("Do lech chuan cua ba cot:")
print(f"  gio_on {dl[0]:.2f}  diem_giua_ky {dl[1]:.2f}"
      f"  chi_tieu {dl[2]:.0f}")
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print("Khoang cach SAU KHI chuan hoa")
print(f"  d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"  d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")
for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(f"  gio {X2[i, 0]:5.1f}  diem {X2[i, 1]:4.1f}  "
          f"khoang cach {khoang[i]:.3f} -> {ket}")
so_qua = int(y[gan].sum())
print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")

Khoang cach KHI CHUA chuan hoa
  d(A, B) =    102.2
  d(A, C) =   3000.0

Do lech chuan cua ba cot:
  gio_on 7.82  diem_giua_ky 2.09  chi_tieu 1692

Khoang cach SAU KHI chuan hoa
  d(A, B) = 3.85
  d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
  gio  10.5  diem  6.4  khoang cach 0.198 -> qua
  gio  12.5  diem  6.1  khoang cach 0.202 -> qua
  gio  14.0  diem  6.4  khoang cach 0.260 -> qua
  gio   9.5  diem  6.1  khoang cach 0.373 -> rot
  gio  10.5  diem  5.8  khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


## 3. Chuẩn hoá (e2)

In [7]:
"""Bước 2: đo hậu quả của việc quên chuẩn hoá trên tập kiểm tra."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Đã sửa đường dẫn từ 'data/sinh_vien_mo_rong.csv' thành 'sinh_vien_mo_rong.csv'
df = pd.read_csv("sinh_vien_mo_rong.csv")
y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)


def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f"Doan bua ca lop qua          : {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa      : {a:.4f}")
print(f"Ba cot, co chuan hoa         : {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa: {c:.4f}")

Doan bua ca lop qua          : 0.6000
Ba cot, khong chuan hoa      : 0.5333
Ba cot, co chuan hoa         : 0.8667
Bo cot chi_tieu, co chuan hoa: 0.9000


## 4. Chọn số láng giềng k (e3)

In [9]:
"""Bước 3: chọn số láng giềng k bằng kiểm định chéo."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Đã sửa đường dẫn từ 'data/hai_vang_trang.csv' thành 'hai_vang_trang.csv'
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉ chênh khoảng
# 1.6 lần, nên ở bộ này không chuẩn hoá cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("    k  tap hoc  tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {k:3d}   {a_hoc:.4f}   {a_kt:.4f}")
print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                              X_tr, y_tr, cv=5).mean()
k_tot = max(diem, key=diem.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = "  <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}  diem trung binh {diem[k]:.4f}{dau}")
print()

# Chọn xong mới mở tập kiểm tra, và chỉ mở một lần
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)
print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Accuracy tren tap hoc va tap kiem tra theo k
    k  tap hoc  tap kiem tra
    1   1.0000   0.8000
    5   0.8756   0.9067
   13   0.8844   0.9200
   31   0.8667   0.8933
  101   0.8311   0.8533

Kiem dinh cheo 5 phan tren tap hoc:
  k =  1  diem trung binh 0.7778
  k =  5  diem trung binh 0.8444
  k =  9  diem trung binh 0.8444
  k = 13  diem trung binh 0.8667  <- tot nhat
  k = 17  diem trung binh 0.8533
  k = 25  diem trung binh 0.8622
  k = 51  diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


## 5. So với các mô hình đã học (e4)

In [12]:
"""Bước 4: đặt KNN cạnh hai mô hình đã học trên dữ liệu có biên cong."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Sửa đường dẫn
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes     ", GaussianNB()),
    ("KNN, k = 13     ", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy tren tap kiem tra, bo hai vang trang")
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    print(f"  {ten}: {accuracy_score(y_te, mh.predict(X_te)):.4f}")

Accuracy tren tap kiem tra, bo hai vang trang
  Hoi quy logistic: 0.8533
  Naive Bayes     : 0.8533
  KNN, k = 13     : 0.9200


## 6. KNN cho bài toán hồi quy (e5)

In [20]:
"""Bước 5: KNN cho bài toán hồi quy, trên dữ liệu giá nhà của bài 1."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

# Sửa đường dẫn tương ứng với tệp gia_nha (1).csv có trong thư mục của bạn
df = pd.read_csv("gia_nha (1).csv")
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f"  KNN, k = 5  : {mae_knn:.4f}")
print(f"  Duong thang : {mae_duong:.4f}")
print()

# Đã thêm cặp dấu ngoặc nhọn {} để sửa lỗi cú pháp tạo DataFrame
can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print("  dien tich    KNN  duong thang")
for dt, a, b in zip(can_moi["dien_tich"], knn.predict(can_moi),
                    duong.predict(can_moi)):
    print(f"  {dt:6.0f} m2  {a:6.2f}  {b:6.2f}")
print()

# Năm láng giềng của căn 250 m2 chính là những căn rộng nhất đã thấy
# Đã thêm cặp dấu ngoặc nhọn {} để sửa lỗi cú pháp tạo DataFrame
_, chi_so = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()
print("Nam lang gieng cua can 250 m2:", ", ".join(f"{v:.1f}" for v in lg))

MAE tren tap kiem tra (ty dong)
  KNN, k = 5  : 0.4030
  Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
  dien tich    KNN  duong thang
      60 m2    4.95    5.12
     100 m2    8.32    8.24
     150 m2    9.32   12.13
     250 m2    9.32   19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


## 7. Bài tập

### Bài tập 1

In [21]:
"""Bài tập 1: tự viết KNN không dùng thư viện (chạy: python baitap04/bai1.py)."""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Tra ve nhan du doan cho MOT mau moi bang bo phieu k lang gieng."""
    khoang = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))  # khoang cach toi moi mau hoc
    gan = np.argsort(khoang)[:k]                           # chi so k mau gan nhat
    nhan, dem = np.unique(y_hoc[gan], return_counts=True)  # dem phieu
    return nhan[np.argmax(dem)]


# Sửa đường dẫn
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

Xa, ya, Xt, yt = (X_tr.to_numpy(float), y_tr.to_numpy(),
                  X_te.to_numpy(float), y_te.to_numpy())

du_doan = np.array([knn_du_doan(Xa, ya, x, 13) for x in Xt])
acc_tu_viet = (du_doan == yt).mean()

mh = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
acc_sklearn = (mh.predict(X_te) == yt).mean()

print(f"Accuracy KNN tu viet    : {acc_tu_viet:.4f}")
print(f"Accuracy KNeighborsClf  : {acc_sklearn:.4f}")
print("Hai con so bang nhau:", np.isclose(acc_tu_viet, acc_sklearn))

Accuracy KNN tu viet    : 0.9200
Accuracy KNeighborsClf  : 0.9200
Hai con so bang nhau: True


### Bài tập 2

In [22]:
"""Bài tập 2: khoảng cách Manhattan so với Euclid."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Sửa đường dẫn
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def chon_k(metric):
    diem = {}
    for k in range(1, 52, 2):
        mh = KNeighborsClassifier(n_neighbors=k, metric=metric)
        diem[k] = cross_val_score(mh, X_tr, y_tr, cv=5).mean()
    k_tot = max(diem, key=diem.get)
    mh = KNeighborsClassifier(n_neighbors=k_tot, metric=metric).fit(X_tr, y_tr)
    return k_tot, diem[k_tot], accuracy_score(y_te, mh.predict(X_te))


print("Khoang cach   k tot nhat  diem CV   accuracy kiem tra")
for ten in ["euclidean", "manhattan"]:
    k, cv, acc = chon_k(ten)
    print(f"{ten:12s}  {k:6d}      {cv:.4f}    {acc:.4f}")

print("\nNhan xet: doi cach do khoang cach chi lam ket qua lech rat it "
      "(xem so tren), vi hai cach do cho thu tu lang gieng gan giong nhau.")

Khoang cach   k tot nhat  diem CV   accuracy kiem tra
euclidean         13      0.8667    0.9200
manhattan         27      0.8667    0.9067

Nhan xet: doi cach do khoang cach chi lam ket qua lech rat it (xem so tren), vi hai cach do cho thu tu lang gieng gan giong nhau.


### Bài tập 3

In [23]:
"""Bài tập 3: đổi đơn vị cột chi_tieu từ nghìn đồng sang triệu đồng."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Sửa đường dẫn
df = pd.read_csv("sinh_vien_mo_rong.csv")
cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
y = df["qua_mon"]

df_trieu = df.copy()
df_trieu["chi_tieu"] = df_trieu["chi_tieu"] / 1000  # nghin dong -> trieu dong


def chay(du_lieu, chuan_hoa):
    X_tr, X_te, y_tr, y_te = train_test_split(
        du_lieu[cot], y, test_size=0.25, random_state=17, stratify=y)
    mh = KNeighborsClassifier(n_neighbors=7)
    if chuan_hoa:
        mh = make_pipeline(StandardScaler(), mh)
    mh.fit(X_tr, y_tr)
    return accuracy_score(y_te, mh.predict(X_te))


print("                          nghin dong   trieu dong")
print(f"Khong chuan hoa          {chay(df, False):10.4f}   {chay(df_trieu, False):10.4f}")
print(f"Co chuan hoa             {chay(df, True):10.4f}   {chay(df_trieu, True):10.4f}")

print("""
Giai thich:
- Khong chuan hoa: chia chi_tieu cho 1000 thi do lech chuan cua no tu ~1692 con
  ~1.7, ngang hai cot kia. Khoang cach khong con bi chi_tieu nuot tron nen ket
  qua tang vot (khac han 0.5333).
- Co chuan hoa: moi cot da duoc dua ve trung binh 0, do lech chuan 1, nen doi
  don vi (phep nhan/chia hang so) khong con anh huong, hai ket qua bang nhau.""")

                          nghin dong   trieu dong
Khong chuan hoa              0.5333       0.8667
Co chuan hoa                 0.8667       0.8667

Giai thich:
- Khong chuan hoa: chia chi_tieu cho 1000 thi do lech chuan cua no tu ~1692 con
  ~1.7, ngang hai cot kia. Khoang cach khong con bi chi_tieu nuot tron nen ket
  qua tang vot (khac han 0.5333).
- Co chuan hoa: moi cot da duoc dua ve trung binh 0, do lech chuan 1, nen doi
  don vi (phep nhan/chia hang so) khong con anh huong, hai ket qua bang nhau.


### Bài tập 4

In [24]:
"""Bài tập 4: láng giềng có trọng số theo khoảng cách."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Sửa đường dẫn
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("   k | uniform: hoc  kiem tra | distance: hoc  kiem tra")
for k in [1, 13, 101]:
    kq = []
    for w in ["uniform", "distance"]:
        mh = KNeighborsClassifier(n_neighbors=k, weights=w).fit(X_tr, y_tr)
        kq += [accuracy_score(y_tr, mh.predict(X_tr)),
               accuracy_score(y_te, mh.predict(X_te))]
    print(f" {k:3d} |        {kq[0]:.4f}  {kq[1]:.4f} |         {kq[2]:.4f}  {kq[3]:.4f}")

print("""
Nhan xet: voi k = 101, phieu cua lang gieng xa duoc giam trong so, lang gieng
gan quyet dinh nhieu hon nen bien bot "cau tha" hon. Doi lai, accuracy tren tap
hoc luon la 1.0000 vi diem hoc tu la lang gieng cach no 0 (trong so rat lon).""")

   k | uniform: hoc  kiem tra | distance: hoc  kiem tra
   1 |        1.0000  0.8000 |         1.0000  0.8000
  13 |        0.8844  0.9200 |         1.0000  0.9333
 101 |        0.8311  0.8533 |         1.0000  0.9067

Nhan xet: voi k = 101, phieu cua lang gieng xa duoc giam trong so, lang gieng
gan quyet dinh nhieu hon nen bien bot "cau tha" hon. Doi lai, accuracy tren tap
hoc luon la 1.0000 vi diem hoc tu la lang gieng cach no 0 (trong so rat lon).


### Bài tập 5

In [25]:
"""Bài tập 5: chọn k cho KNN hồi quy bằng kiểm định chéo."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

# Sửa đường dẫn tương ứng với tệp gia_nha (1).csv có trong thư mục của bạn
df = pd.read_csv("gia_nha (1).csv")
X = df[["dien_tich"]]
y = df["gia"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

diem = {}
for k in range(1, 21):
    s = cross_val_score(KNeighborsRegressor(n_neighbors=k), X_tr, y_tr,
                        cv=5, scoring="neg_mean_absolute_error")
    diem[k] = -s.mean()  # diem tra ve mang dau am nen doi dau lai thanh MAE

for k, v in diem.items():
    print(f"  k = {k:2d}  MAE kiem dinh cheo = {v:.4f}")

k_tot = min(diem, key=diem.get)
knn = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print(f"\nk tot nhat = {k_tot}")
print(f"MAE kiem tra KNN (k={k_tot}) : {mae_knn:.4f}")
print(f"MAE kiem tra duong thang  : {mae_duong:.4f}")
print("KNN vuot duong thang!" if mae_knn < mae_duong
      else "KNN van chua vuot duoc duong thang (quan he that gan nhu thang).")

  k =  1  MAE kiem dinh cheo = 0.4164
  k =  2  MAE kiem dinh cheo = 0.3498
  k =  3  MAE kiem dinh cheo = 0.3714
  k =  4  MAE kiem dinh cheo = 0.3618
  k =  5  MAE kiem dinh cheo = 0.3816
  k =  6  MAE kiem dinh cheo = 0.4002
  k =  7  MAE kiem dinh cheo = 0.3990
  k =  8  MAE kiem dinh cheo = 0.4098
  k =  9  MAE kiem dinh cheo = 0.4171
  k = 10  MAE kiem dinh cheo = 0.4365
  k = 11  MAE kiem dinh cheo = 0.4500
  k = 12  MAE kiem dinh cheo = 0.4695
  k = 13  MAE kiem dinh cheo = 0.4992
  k = 14  MAE kiem dinh cheo = 0.5153
  k = 15  MAE kiem dinh cheo = 0.5290
  k = 16  MAE kiem dinh cheo = 0.5595
  k = 17  MAE kiem dinh cheo = 0.5748
  k = 18  MAE kiem dinh cheo = 0.5891
  k = 19  MAE kiem dinh cheo = 0.6130
  k = 20  MAE kiem dinh cheo = 0.6254

k tot nhat = 2
MAE kiem tra KNN (k=2) : 0.4867
MAE kiem tra duong thang  : 0.2578
KNN van chua vuot duoc duong thang (quan he that gan nhu thang).


### Bài tập 6

In [26]:
"""Bài tập 6: đo thời gian .fit và .predict của KNN theo cỡ dữ liệu."""

import time

import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X_moi = rng.random((1000, 2))

print("   so mau    fit (giay)   predict 1000 mau (giay)")
for n in [1000, 10000, 100000]:
    X = rng.random((n, 2))
    y = rng.integers(0, 2, n)
    # algorithm="brute" de thay ro viec so sanh voi toan bo du lieu
    mh = KNeighborsClassifier(n_neighbors=5, algorithm="brute")

    t0 = time.perf_counter()
    mh.fit(X, y)
    t_fit = time.perf_counter() - t0

    t0 = time.perf_counter()
    mh.predict(X_moi)
    t_pre = time.perf_counter() - t0

    print(f"  {n:7d}   {t_fit:10.5f}   {t_pre:10.5f}")

print("""
Giai thich: .fit gan nhu tuc thi vi chi cat du lieu vao bo nho, con .predict
cham dan theo so mau vi phai tinh khoang cach toi toan bo du lieu hoc. Moi
phep tinh bi don het sang luc du doan, do do KNN la mo hinh "hoc luoi".""")


   so mau    fit (giay)   predict 1000 mau (giay)
     1000      0.00124      0.00843
    10000      0.00104      0.04460
   100000      0.00868      0.42090

Giai thich: .fit gan nhu tuc thi vi chi cat du lieu vao bo nho, con .predict
cham dan theo so mau vi phai tinh khoang cach toi toan bo du lieu hoc. Moi
phep tinh bi don het sang luc du doan, do do KNN la mo hinh "hoc luoi".
